Step 1: Import Libraries

In [0]:
from pyspark.sql.functions import *

Step 2: Read Bronze Tables

In [0]:
sales_df = spark.table(
    "retail_analytics_catalog.bronze.sales_raw"
)

customer_df = spark.table(
    "retail_analytics_catalog.bronze.customers_raw"
)

product_df = spark.table(
    "retail_analytics_catalog.bronze.products_raw"
)

Step 3: Null Validation

In [0]:
null_records = sales_df.filter(
    col("customer_id").isNull() |
    col("product_id").isNull()
)

display(null_records)

Step 4: Negative and Zero Quantity Validation

In [0]:
invalid_quantity = sales_df.filter(
    col("quantity") <= 0
)

display(invalid_quantity)

Step 5: Invalid Customer Validation

In [0]:
invalid_customer = (
    sales_df
    .join(
        customer_df,
        "customer_id",
        "left_anti"
    )
)

display(invalid_customer)

Step 6: Invalid Product Validation

In [0]:
invalid_product = (
    sales_df
    .join(
        product_df,
        "product_id",
        "left_anti"
    )
)

display(invalid_product)

Step 7: Duplicate Validation

In [0]:
duplicate_records = (
    sales_df
    .groupBy("order_id")
    .count()
    .filter(col("count") > 1)
)

display(duplicate_records)

Step 8: Load Rejected Records

In [0]:
null_records \
.withColumn(
    "reason",
    lit("Null Customer/Product")
) \
.withColumn(
    "load_time",
    current_timestamp()
) \
.write.mode("append") \
.saveAsTable(
    "retail_analytics_catalog.silver.rejected_records"
)

In [0]:
invalid_quantity \
.withColumn(
    "reason",
    lit("Invalid Quantity")
) \
.withColumn(
    "load_time",
    current_timestamp()
) \
.write.mode("append") \
.saveAsTable(
    "retail_analytics_catalog.silver.rejected_records"
)

In [0]:
invalid_customer \
.withColumn(
    "reason",
    lit("Invalid Customer")
) \
.withColumn(
    "load_time",
    current_timestamp()
) \
.write.mode("append") \
.saveAsTable(
    "retail_analytics_catalog.silver.rejected_records"
)

In [0]:
invalid_product \
.withColumn(
    "reason",
    lit("Invalid Product")
) \
.withColumn(
    "load_time",
    current_timestamp()
) \
.write.mode("append") \
.saveAsTable(
    "retail_analytics_catalog.silver.rejected_records"
)

Step 9: Create Valid Records Dataset

In [0]:
valid_sales = sales_df.filter(
    col("customer_id").isNotNull()
    & col("product_id").isNotNull()
    & (col("quantity") > 0)
)

Step 10: Remove Invalid Customers

In [0]:
valid_sales = (
    valid_sales
    .join(
        customer_df,
        "customer_id",
        "inner"
    )
)

Step 11: Remove Invalid Products

In [0]:
valid_sales = (
    valid_sales
    .join(
        product_df.select("product_id"),
        "product_id",
        "inner"
    )
)

Step 12: Remove Duplicates

In [0]:
valid_sales = valid_sales.dropDuplicates(
    ["order_id"]
)

Step 13: Create Temporary View

In [0]:
valid_sales.createOrReplaceTempView(
    "valid_sales"
)

Validation Summary

In [0]:
print("Total Records:", sales_df.count())

print("Valid Records:", valid_sales.count())

print(
    "Rejected Records:",
    sales_df.count() - valid_sales.count()
)

Verify Rejected Records

In [0]:
display(
    spark.table(
        "retail_analytics_catalog.silver.rejected_records"
    )
)